In [1]:
import pandas as pd
from evidently import Dataset
from evidently import DataDefinition
from evidently import Report
from evidently.presets import TextEvals
from evidently.tests import lte, gte, eq
from evidently.descriptors import LLMEval, TestSummary, DeclineLLMEval, Sentiment, TextLength, IncludesWords
from evidently.llm.templates import BinaryClassificationPromptTemplate

In [2]:
from evidently.ui.workspace import CloudWorkspace

## Configuration credential

In [5]:
# Configuration for the credential
import os
from dotenv import load_dotenv

ENV_PATH = ".env"
load_dotenv(ENV_PATH)

False

## Create project on evidently AI

In [6]:
ws = CloudWorkspace(token=os.getenv("EVIDENTLY_TOKEN"), url="https://app.evidently.cloud")

## Create a Project within your Organization, or connect to an existing Project:

In [ ]:
try:
    project = ws.create_project(name="LLM Audit", org_id="019d6885-06c6-7601-9a3e-2298bed991f4")
    project.description = "This project is for auditing the performmance of LLMs in the context of a banking project. We will be using the Evidently AI platform to evaluate the performance of the LLMs " \
    "and identify any potential issues or areas for improvement."
    project.save()
except Exception as e:
    if project is not None:
        project = ws.get_project_by_name("LLM Audit")
        print(f"Project already exists: {project.name}")

## Prepare the dataset

In [17]:
data = [
        ["What is the fraud detection rate in the dataset?", "Fraud detection in banking uses ML classifiers. Precision-recall tradeoffs are critical.",
        "The dataset shows a fraud label distribution. High precision minimises false positives."],
        ["Which ML model performs best for credit default prediction?",
        "XGBoost and Random Forest outperform logistic regression on imbalanced datasets.",
        "XGBoost typically achieves the highest ROC-AUC for credit default tasks."],
        ["How should missing values be handled in operational banking data?",
        "Imputation must preserve regulatory compliance and not introduce synthetic bias.",
        "Use median for numerical and mode for categorical. Flag missingness as a binary indicator."],
        ["What is the churn rate and how can it be reduced?",
        "Customer churn in retail banking averages 15-20% annually.",
        "Churn column shows the distribution. Targeted retention can reduce churn by 10-15%."],
        ["Explain the class imbalance issue in fraud detection.",
        "Fraud datasets are heavily imbalanced with fraud rates below 1-2%.",
        "Requires resampling or class-weight adjustment. Use F1 and PR-AUC, not accuracy."],
]
columns = ["question", "context", "answer"]

pd.set_option('display.max_colwidth', None)
eval_df = pd.DataFrame(data, columns=columns)
eval_df

,question,context,answer
0,What is the fraud detection rate in the dataset?,Fraud detection in banking uses ML classifiers. Precision-recall tradeoffs are critical.,The dataset shows a fraud label distribution. High precision minimises false positives.
1,Which ML model performs best for credit default prediction?,XGBoost and Random Forest outperform logistic regression on imbalanced datasets.,XGBoost typically achieves the highest ROC-AUC for credit default tasks.
2,How should missing values be handled in operational banking data?,Imputation must preserve regulatory compliance and not introduce synthetic bias.,Use median for numerical and mode for categorical. Flag missingness as a binary indicator.
3,What is the churn rate and how can it be reduced?,Customer churn in retail banking averages 15-20% annually.,Churn column shows the distribution. Targeted retention can reduce churn by 10-15%.
4,Explain the class imbalance issue in fraud detection.,Fraud datasets are heavily imbalanced with fraud rates below 1-2%.,"Requires resampling or class-weight adjustment. Use F1 and PR-AUC, not accuracy."


## Auditing the LLMs for evaluation

* ###  Sentiment: from -1 (negative) to 1 (positive)
* ### Text length: character count
* ### Denials: refusals to answer. This uses an LLM-as-a-judge with built-in prompt.

### Each evaluation is a descriptor. It adds a new score or label to each row in your dataset. For LLM-as-a-judge, we’ll use OpenAI GPT-4o mini. Set OpenAI key as an environment variable:

In [21]:
import os

os.environ["GROQ_API_KEY"] = os.getenv("GROQ_API_KEY")

eval_dataset = Dataset.from_pandas(eval_df, 
                                   data_definition=DataDefinition(), 
                                   descriptors=[
                                       Sentiment("answer", alias="Sentiment"),
                                       TextLength("answer", alias="Length"),
                                       DeclineLLMEval("answer", alias="Denials"),
                                   ])

# Preview the dataset with descriptors
eval_dataset.as_dataframe()

LLMRateLimitError: Error code: 429 - {'error': {'message': 'You exceeded your current quota, please check your plan and billing details. For more information on this error, read the docs: https://platform.openai.com/docs/guides/error-codes/api-errors.', 'type': 'insufficient_quota', 'param': None, 'code': 'insufficient_quota'}}